# nanoHUB-QE Advanced Tutorial (espresso-7.1)

This notebook demonstrates a more complex workflow using `nanohubqe` with explicit `espresso-7.1` loading: SCF + DOS + bands + phonons, plotting, and remote `submit` command generation.

In [ ]:
import nanohubqe
from nanohubqe import __version__
print('nanohubqe version:', __version__)


## 1. Load Quantum ESPRESSO 7.1 on nanoHUB

This tutorial intentionally loads `espresso-7.1` explicitly.

In [ ]:
from nanohubqe import list_available_modules, load_quantum_espresso

espresso_like = list_available_modules('espresso')
print('ESPRESSO-like modules:', espresso_like[:20])

loaded = load_quantum_espresso('espresso-7.1')
print('Loaded module:', loaded)


## 2. Build and run a complex workflow (SCF + DOS + Bands + Phonons)

For a fast demo, this uses moderate cutoffs and meshes. Increase these for production-quality results.

In [ ]:
from pathlib import Path
from shutil import which
from nanohubqe import bulk_electronic_phonon_workflow

sim = bulk_electronic_phonon_workflow(
    symbol='Si',
    structure='diamond',
    title='Si advanced electronic + phonon demo',
    prefix='si_adv',
    pseudo_dir='./pseudo',
    outdir='./tmp',
    pseudo_file='Si.UPF',
    ecutwfc=20.0,
    ecutrho=160.0,
    nbnd=10,
    scf_k_points=(4, 4, 4, 0, 0, 0),
    include_dos=True,
    dos_emin=-8.0,
    dos_emax=12.0,
    dos_deltae=0.05,
    include_bands=True,
    bands_k_path=[
        (0.5, 0.5, 0.5, 20.0),
        (0.0, 0.0, 0.0, 20.0),
        (1.0, 0.0, 0.0, 20.0),
        (1.0, 0.5, 0.0, 20.0),
        (0.75, 0.75, 0.0, 20.0),
        (0.0, 0.0, 0.0, 20.0),
    ],
    include_plotband=False,
    include_phonon=True,
    phonon_q_grid=(2, 2, 2),
    phonon_q_path=[
        (0.0, 0.0, 0.0),
        (0.5, 0.0, 0.0),
        (0.5, 0.5, 0.0),
        (0.5, 0.5, 0.5),
        (0.0, 0.0, 0.0),
    ],
    phonon_q_num=31,
)

workdir = Path('runs/advanced-si-espresso71')
workdir.mkdir(parents=True, exist_ok=True)

pseudo_status = sim.prepare_pseudopotentials(workdir=workdir)
for item in pseudo_status:
    print(f'{item.pseudo_file}: {item.action} -> {item.target_path}')

required = ['pw.x', 'dos.x', 'bands.x', 'ph.x', 'q2r.x', 'matdyn.x']
missing = [exe for exe in required if which(exe) is None]
dry_run = bool(missing)
if dry_run:
    print('Missing executables:', ', '.join(missing))
    print('Running in dry_run=True mode (command generation only).')
else:
    print('Running full workflow with local QE executables...')

sim.run(workdir=workdir, dry_run=dry_run)
for step_name, result in sim.results.items():
    print(step_name, 'rc=', result.returncode, 'ok=', result.ok)


## 3. Inspect outputs tracked by the workflow


In [ ]:
print('Executed steps:', list(sim.results))
print('JSON output record:', workdir / 'workflow_outputs.json')

for step_name in sim.results:
    result = sim.step_result(step_name)
    discovered = [path.name for path in result.discovered_outputs]
    print(step_name, '->', discovered)


## 4. Plot results (matplotlib + plotly backends)


In [ ]:
from IPython.display import display

if dry_run:
    print('Dry run mode generated commands only. Re-run with dry_run=False for plottable data.')
else:
    ax = sim.plot_total_energy(backend='matplotlib')
    display(ax.figure)
    display(sim.plot_dos(backend='plotly'))
    display(sim.plot_bands(backend='plotly', k_labels=['L', 'G', 'X', 'W', 'K', 'G']))
    display(sim.plot_phonon_dispersion(backend='plotly'))


## 5. Remote execution example with HUBzero `submit`

This executes the advanced workflow remotely, waits for each step, and syncs outputs so local plotting can be used afterwards.



In [ ]:
from nanohubqe import QERunner, SubmitConfig, bulk_electronic_phonon_workflow

remote_sim = bulk_electronic_phonon_workflow(
    symbol='Si',
    structure='diamond',
    prefix='si_adv_remote',
    pseudo_dir='./pseudo',
    outdir='./tmp',
    pseudo_file='Si.UPF',
    ecutwfc=20.0,
    ecutrho=160.0,
    nbnd=10,
    scf_k_points=(4, 4, 4, 0, 0, 0),
    include_dos=True,
    include_bands=True,
    include_plotband=False,
    include_phonon=True,
)

remote_workdir = Path('runs/advanced-si-espresso71-remote')
remote_workdir.mkdir(parents=True, exist_ok=True)
remote_sim.prepare_pseudopotentials(workdir=remote_workdir)

submit_cfg = SubmitConfig(
    venue='nanohub',
    nodes=8,
    walltime='01:00:00',
    manager='espresso-7.1_mpi-cleanup_pw',
    run_name='si-advanced-espresso71',
    executable_prefix='espresso-7.1',
    env={'ESPRESSO_PSEUDO': './pseudo'},
)

remote_runner = QERunner(default_backend='submit')

remote_sim.run_submit(
    workdir=remote_workdir,
    runner=remote_runner,
    submit_config=submit_cfg,
    dry_run=False,
    wait=True,
    sync_outputs=True,
    poll_interval=20.0,
    wait_timeout=7200.0,
)

for step_name, result in remote_sim.results.items():
    print(step_name, 'rc=', result.returncode, 'status=', result.remote_status, 'synced=', result.outputs_synced)

